# Notebook 6: End-to-End Workflow

STAC search → sign → open as chunked xarray → compute NDVI → reduce over space and (optionally) time.

**Dependencies:** `pystac-client`, `planetary-computer`, `rioxarray`, `xarray`

In [ ]:
import pystac_client
import planetary_computer
import rioxarray
import xarray as xr

## 1. STAC search and sign

In [ ]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1"
)
search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=[-122.4, 37.6, -122.2, 37.8],
    datetime="2023-06-01/2023-06-30",
    max_items=3,
)
items = [planetary_computer.sign(it) for it in search.items()]
print(f"Signed {len(items)} items")

## 2. Open bands as chunked xarray per item

Helper: open B04 and B08 for one item, return a Dataset with a `time` coordinate.

In [ ]:
def open_bands(item, band_ids, chunks=None):
    bands = {}
    for bid in band_ids:
        href = item.assets[bid].href
        da = rioxarray.open_rasterio(href, chunks=chunks or "auto").squeeze("band", drop=True)
        bands[bid] = da
    ds = xr.Dataset(bands)
    ds = ds.assign_coords(time=item.datetime)
    return ds

chunks = {"x": 1024, "y": 1024}
scenes = [open_bands(it, ["B04", "B08"], chunks) for it in items]

## 3. Concatenate over time and compute NDVI

In [ ]:
combined = xr.concat(scenes, dim="time")
combined["ndvi"] = (
    (combined["B08"] - combined["B04"]) / (combined["B08"] + combined["B04"])
)
combined

## 4. Reduce: mean NDVI over space (per time)

Result: one value per date. This triggers Dask to read only the chunks needed.

In [ ]:
ts = combined["ndvi"].mean(dim=["x", "y"])
ts_computed = ts.compute()
ts_computed

## 5. Optional: plot time series

In [ ]:
import matplotlib.pyplot as plt

ts_computed.plot(marker="o")
plt.title("Mean NDVI over area (Sentinel-2 L2A)")
plt.ylabel("NDVI")
plt.tight_layout()
plt.show()